In [4]:
import folium


In [5]:
# Approximate center of the Gila River Indian Community
latitude = 33.15
longitude = -111.83


In [6]:
# Create map
m = folium.Map(
    location=[latitude, longitude],
    zoom_start=11,
    tiles="OpenStreetMap"
)

In [7]:
# Add marker
folium.Marker(
    [latitude, longitude],
    popup="Gila River Indian Community",
    tooltip="Gila River Indian Community"
).add_to(m)

In [8]:
# Optional: draw approximate community boundary area
boundary_points = [
    [33.30, -112.05],
    [33.30, -111.60],
    [33.00, -111.60],
    [33.00, -112.05],
    [33.30, -112.05]
]

In [9]:
folium.Polygon(
    locations=boundary_points,
    color="blue",
    weight=2,
    fill=True,
    fill_opacity=0.2,
    popup="Approximate Community Area"
).add_to(m)

In [10]:
# Save map
m.save("gila_river_community_map.html")

print("Map saved as gila_river_community_map.html")

Map saved as gila_river_community_map.html


In [1]:
import osmnx as ox
import folium

In [2]:
place = "Gila River Indian Community, Arizona, USA"

gdf = ox.geocode_to_gdf(place)


In [3]:
center = [
    gdf.geometry.centroid.y.iloc[0],
    gdf.geometry.centroid.x.iloc[0]
]

/tmp/ipykernel_13127/3642108053.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf.geometry.centroid.y.iloc[0],
/tmp/ipykernel_13127/3642108053.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf.geometry.centroid.x.iloc[0]


In [4]:
m = folium.Map(location=center, zoom_start=11)


In [5]:
folium.GeoJson(
    gdf.geometry,
    name="Gila River Indian Community"
).add_to(m)

In [6]:
m.save("gila_river_osm_boundary.html")

print("Boundary map saved.")

Boundary map saved.


In [7]:
import osmnx as ox
import geopandas as gpd
import folium

In [8]:
PLACE = "Gila River Indian Community, Arizona, USA"

In [9]:
print("Downloading boundary...")

boundary = ox.geocode_to_gdf(PLACE)

center_lat = boundary.geometry.centroid.y.iloc[0]
center_lon = boundary.geometry.centroid.x.iloc[0]


/tmp/ipykernel_13127/2118963031.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = boundary.geometry.centroid.y.iloc[0]
/tmp/ipykernel_13127/2118963031.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = boundary.geometry.centroid.x.iloc[0]


In [12]:
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="OpenStreetMap"
)

In [13]:
folium.GeoJson(
    boundary,
    name="Community Boundary",
    style_function=lambda x: {
        "color": "red",
        "weight": 3,
        "fillOpacity": 0.05
    }
).add_to(m)

In [14]:

print("Downloading buildings...")

tags = {"building": True}

buildings = ox.features_from_polygon(
    boundary.geometry.iloc[0],
    tags
)

if not buildings.empty:

    buildings = buildings[
        buildings.geometry.type.isin(
            ["Polygon", "MultiPolygon"]
        )
    ]

    folium.GeoJson(
        buildings,
        name="Buildings",
        style_function=lambda x: {
            "color": "#666666",
            "weight": 1,
            "fillColor": "#999999",
            "fillOpacity": 0.4
        }
    ).add_to(m)

    print(f"Buildings loaded: {len(buildings)}")

Buildings loaded: 3192


In [15]:
print("Downloading water features...")

water_tags = {
    "natural": ["water"],
    "waterway": True
}

water = ox.features_from_polygon(
    boundary.geometry.iloc[0],
    water_tags
)

if not water.empty:

    folium.GeoJson(
        water,
        name="Water Features",
        style_function=lambda x: {
            "color": "blue",
            "weight": 2,
            "fillColor": "lightblue",
            "fillOpacity": 0.5
        }
    ).add_to(m)

    print(f"Water features loaded: {len(water)}")


Water features loaded: 776


In [16]:
print("Downloading roads...")

graph = ox.graph_from_polygon(
    boundary.geometry.iloc[0],
    network_type="drive"
)

edges = ox.graph_to_gdfs(
    graph,
    nodes=False
)

folium.GeoJson(
    edges,
    name="Roads",
    style_function=lambda x: {
        "color": "black",
        "weight": 1
    }
).add_to(m)

In [17]:
folium.LayerControl(collapsed=False).add_to(m)

In [18]:

output_file = "gila_river_interactive_map.html"

m.save(output_file)

print(f"Map saved to: {output_file}")

Map saved to: gila_river_interactive_map.html
